# MLOps and Production ML
## Senior ML Interview Preparation

This notebook covers the full lifecycle of deploying and maintaining machine learning models in production. These topics are increasingly tested in senior ML engineer and ML scientist interviews at top technology companies.

### What You Will Learn
- Experiment tracking with MLflow (and manual equivalents)
- Reproducible ML pipelines with scikit-learn
- Data drift and concept drift detection
- Model deployment patterns (REST API, batch, A/B testing)
- Production best practices: feature stores, versioning, rollback

### Why This Matters in Interviews
Senior ML roles expect you to go beyond model accuracy. Interviewers test whether you understand the full production lifecycle: reproducibility, monitoring, reliability, and iteration speed.

---
**Sections:**
1. Experiment Tracking with MLflow
2. ML Pipelines and Reproducibility
3. Model Monitoring After Deployment
4. Model Deployment Patterns
5. Production ML Best Practices + 15 Interview Q&A

In [ ]:
# Core imports used throughout this notebook
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
import os
import json
import time
import hashlib
import joblib
from datetime import datetime
from pathlib import Path
from scipy import stats
from sklearn.datasets import load_wine, make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from sklearn.impute import SimpleImputer

warnings.filterwarnings('ignore')
np.random.seed(42)

print('All core imports successful.')
print(f'NumPy: {np.__version__}')
print(f'Pandas: {pd.__version__}')
print(f'Scikit-learn: {__import__("sklearn").__version__}')

---
## Section 1 -- Experiment Tracking with MLflow

Experiment tracking solves a core pain point: after running 50 experiments, which hyperparameters produced the best model? Which preprocessing steps were used? What was the training data version?

**MLflow** is the industry-standard open-source solution. It provides:
- **Tracking**: log parameters, metrics, artifacts per run
- **Projects**: reproducible ML code packaging
- **Models**: standardized model format
- **Registry**: centralized model lifecycle management

### Interview Context
*"How do you keep track of your experiments?"* is a standard senior ML question. The expected answer demonstrates systematic experiment management, not just ad-hoc scripts.

In [ ]:
# Try to import MLflow; fall back to a manual tracker if not installed
try:
    import mlflow
    import mlflow.sklearn
    MLFLOW_AVAILABLE = True
    print(f'MLflow version: {mlflow.__version__}')
    print('Using real MLflow tracking.')
except ImportError:
    MLFLOW_AVAILABLE = False
    print('MLflow not installed. Using manual experiment tracker (equivalent concept).')
    print('To install: pip install mlflow')

In [ ]:
# ============================================================
# Manual Experiment Tracker -- identical concept to MLflow
# This is the fallback AND a great way to understand what
# MLflow does under the hood.
# ============================================================

class ManualExperimentTracker:
    """Lightweight experiment tracker that mirrors MLflow's API."""

    def __init__(self, experiment_name, tracking_dir='./mlruns_manual'):
        self.experiment_name = experiment_name
        self.tracking_dir = Path(tracking_dir) / experiment_name
        self.tracking_dir.mkdir(parents=True, exist_ok=True)
        self.active_run = None
        self.runs = []

    def start_run(self, run_name=None):
        run_id = hashlib.md5(f'{time.time()}'.encode()).hexdigest()[:8]
        self.active_run = {
            'run_id': run_id,
            'run_name': run_name or f'run_{run_id}',
            'start_time': datetime.now().isoformat(),
            'params': {},
            'metrics': {},
            'tags': {}
        }
        return self

    def __enter__(self):
        return self

    def __exit__(self, *args):
        self.end_run()

    def log_param(self, key, value):
        self.active_run['params'][key] = value

    def log_params(self, params_dict):
        self.active_run['params'].update(params_dict)

    def log_metric(self, key, value):
        self.active_run['metrics'][key] = value

    def log_metrics(self, metrics_dict):
        self.active_run['metrics'].update(metrics_dict)

    def set_tag(self, key, value):
        self.active_run['tags'][key] = value

    def end_run(self):
        if self.active_run:
            self.active_run['end_time'] = datetime.now().isoformat()
            run_file = self.tracking_dir / f"{self.active_run['run_id']}.json"
            with open(run_file, 'w') as f:
                json.dump(self.active_run, f, indent=2)
            self.runs.append(self.active_run.copy())
            self.active_run = None

    def get_all_runs(self):
        """Load all run records and return as a DataFrame."""
        records = []
        for f in self.tracking_dir.glob('*.json'):
            with open(f) as fh:
                records.append(json.load(fh))
        rows = []
        for r in records:
            row = {'run_id': r['run_id'], 'run_name': r['run_name']}
            row.update({f'param.{k}': v for k, v in r['params'].items()})
            row.update({f'metric.{k}': v for k, v in r['metrics'].items()})
            rows.append(row)
        return pd.DataFrame(rows)

print('ManualExperimentTracker defined.')
print("This mirrors MLflow's core API: log_param, log_metric, log_params, log_metrics.")

In [ ]:
# ============================================================
# Load Wine Quality Dataset
# ============================================================
wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = wine.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Wine dataset: {X.shape[0]} samples, {X.shape[1]} features, {len(np.unique(y))} classes')
print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')
print(f'Features: {list(wine.feature_names)[:5]}...')
print(f'Classes: {wine.target_names}')

In [ ]:
# ============================================================
# Run Multiple Experiments and Log Everything
# ============================================================

tracker = ManualExperimentTracker('wine_quality_experiment')

experiment_configs = [
    {'model': 'LogisticRegression', 'C': 0.1,  'max_iter': 1000, 'solver': 'lbfgs'},
    {'model': 'LogisticRegression', 'C': 1.0,  'max_iter': 1000, 'solver': 'lbfgs'},
    {'model': 'LogisticRegression', 'C': 10.0, 'max_iter': 1000, 'solver': 'lbfgs'},
    {'model': 'RandomForest',       'n_estimators': 50,  'max_depth': 5},
    {'model': 'RandomForest',       'n_estimators': 100, 'max_depth': 10},
    {'model': 'RandomForest',       'n_estimators': 200, 'max_depth': None},
]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

results = []

for cfg in experiment_configs:
    run_name = f"{cfg['model']}_run_{len(results)+1}"
    tracker.start_run(run_name=run_name)

    # Log hyperparameters
    tracker.log_params(cfg)
    tracker.log_param('dataset', 'wine_quality_sklearn')
    tracker.log_param('test_size', 0.2)
    tracker.log_param('random_state', 42)
    tracker.log_param('scaler', 'StandardScaler')

    # Build and train model
    t0 = time.time()
    if cfg['model'] == 'LogisticRegression':
        model = LogisticRegression(C=cfg['C'], max_iter=cfg['max_iter'],
                                   solver=cfg['solver'], multi_class='auto')
        model.fit(X_train_scaled, y_train)
        preds      = model.predict(X_test_scaled)
        preds_prob = model.predict_proba(X_test_scaled)
    else:
        model = RandomForestClassifier(
            n_estimators=cfg['n_estimators'],
            max_depth=cfg['max_depth'],
            random_state=42
        )
        model.fit(X_train_scaled, y_train)
        preds      = model.predict(X_test_scaled)
        preds_prob = model.predict_proba(X_test_scaled)

    train_time = time.time() - t0

    # Compute metrics
    acc = accuracy_score(y_test, preds)
    auc = roc_auc_score(y_test, preds_prob, multi_class='ovr', average='macro')
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='accuracy')

    # Log metrics
    tracker.log_metrics({
        'test_accuracy':   round(acc, 4),
        'test_roc_auc':    round(auc, 4),
        'cv_mean':         round(cv_scores.mean(), 4),
        'cv_std':          round(cv_scores.std(), 4),
        'train_time_sec':  round(train_time, 3)
    })
    tracker.set_tag('model_class', cfg['model'])
    tracker.end_run()

    results.append({
        'run_name':   run_name,
        'config':     cfg,
        'accuracy':   acc,
        'roc_auc':    auc,
        'cv_mean':    cv_scores.mean(),
        'best_model': model
    })
    print(f'{run_name:40s}  acc={acc:.4f}  auc={auc:.4f}  cv={cv_scores.mean():.4f}+/-{cv_scores.std():.4f}')

print('\nAll experiment runs logged.')

In [ ]:
# ============================================================
# Compare Runs -- like MLflow UI
# ============================================================
run_df = tracker.get_all_runs()
print('Experiment run comparison:')
print(run_df.filter(regex='run_name|metric').sort_values('metric.test_accuracy', ascending=False).to_string(index=False))

best_idx  = run_df['metric.test_accuracy'].idxmax()
best_run  = run_df.loc[best_idx, 'run_name']
best_acc  = run_df.loc[best_idx, 'metric.test_accuracy']
print(f'\nBest run: {best_run} with test accuracy = {best_acc}')

In [ ]:
# ============================================================
# Visualize Experiment Results
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

names    = [r['run_name'].replace('_run_', '\n#') for r in results]
accs     = [r['accuracy'] for r in results]
cv_means = [r['cv_mean']  for r in results]
colors   = ['steelblue' if 'Logistic' in n else 'darkorange' for n in names]

axes[0].barh(names, accs, color=colors, edgecolor='white', linewidth=0.5)
axes[0].set_xlabel('Test Accuracy')
axes[0].set_title('Test Accuracy by Run')
axes[0].axvline(0.9, color='red', linestyle='--', linewidth=1, label='0.90 threshold')
axes[0].legend(fontsize=8)

axes[1].barh(names, cv_means, color=colors, edgecolor='white', linewidth=0.5)
axes[1].set_xlabel('Cross-Validation Mean Accuracy')
axes[1].set_title('CV Accuracy by Run')
axes[1].axvline(0.9, color='red', linestyle='--', linewidth=1)

plt.suptitle('MLflow-Style Experiment Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print('Blue = LogisticRegression, Orange = RandomForest')

### Model Registry Concept

MLflow's **Model Registry** is a centralized store for managing model lifecycle:

```
Registered Model: wine_quality_classifier
  Version 1  (Staging)   -> LogisticRegression C=1.0, acc=0.917
  Version 2  (Production) -> RandomForest n=200, acc=0.944
  Version 3  (Staging)   -> GBM, testing in progress
```

**Lifecycle stages:**
- **None**: freshly logged
- **Staging**: under evaluation / QA
- **Production**: serving live traffic
- **Archived**: retired

**With real MLflow you would:**
```python
# Register a model from a run
mlflow.register_model(model_uri=f'runs:/{run_id}/model', name='wine_classifier')

# Transition stage
client = mlflow.tracking.MlflowClient()
client.transition_model_version_stage(
    name='wine_classifier', version=1, stage='Production'
)

# Load production model anywhere
model = mlflow.pyfunc.load_model('models:/wine_classifier/Production')
```

The registry decouples **training** from **deployment** -- a key architectural principle.

In [ ]:
# ============================================================
# Manual Model Registry -- mirrors MLflow Model Registry
# ============================================================

class SimpleModelRegistry:
    """In-memory model registry with versioning and lifecycle stages."""

    VALID_STAGES = {'None', 'Staging', 'Production', 'Archived'}

    def __init__(self, name):
        self.name     = name
        self.versions = []  # list of dicts
        self._version_counter = 0

    def register(self, model, run_name, metrics, description=''):
        self._version_counter += 1
        entry = {
            'version':     self._version_counter,
            'model':       model,
            'run_name':    run_name,
            'metrics':     metrics,
            'stage':       'None',
            'created_at':  datetime.now().isoformat(),
            'description': description
        }
        self.versions.append(entry)
        print(f'Registered: {self.name} v{self._version_counter} (stage=None)')
        return self._version_counter

    def transition_stage(self, version, new_stage):
        if new_stage not in self.VALID_STAGES:
            raise ValueError(f'Stage must be one of {self.VALID_STAGES}')
        for v in self.versions:
            if v['version'] == version:
                old = v['stage']
                v['stage'] = new_stage
                print(f'{self.name} v{version}: {old} -> {new_stage}')
                return
        raise ValueError(f'Version {version} not found')

    def get_production_model(self):
        prod = [v for v in self.versions if v['stage'] == 'Production']
        if not prod:
            raise RuntimeError('No model in Production stage')
        return prod[-1]['model']  # latest production version

    def summary(self):
        print(f'\nModel Registry: {self.name}')
        print(f'{"Version":<10}{"Stage":<14}{"Run Name":<40}{"Accuracy":<10}')
        print('-' * 74)
        for v in self.versions:
            acc = v['metrics'].get('accuracy', 'N/A')
            print(f"{v['version']:<10}{v['stage']:<14}{v['run_name']:<40}{str(acc):<10}")


# Populate registry with our experiment results
registry = SimpleModelRegistry('wine_quality_classifier')
for r in results:
    registry.register(
        model=r['best_model'],
        run_name=r['run_name'],
        metrics={'accuracy': round(r['accuracy'], 4), 'cv_mean': round(r['cv_mean'], 4)},
        description=str(r['config'])
    )

registry.transition_stage(1, 'Archived')
registry.transition_stage(5, 'Staging')
registry.transition_stage(6, 'Production')
registry.summary()

---
## Section 2 -- ML Pipelines and Reproducibility

### The Data Leakage Problem

One of the most dangerous and common mistakes in ML:

```
BAD (data leakage):
    scaler.fit(X_all)          # scaler sees test data!
    X_scaled = scaler.transform(X_all)
    X_train, X_test = train_test_split(X_scaled)

GOOD (no leakage):
    X_train, X_test = train_test_split(X_all)
    scaler.fit(X_train)         # scaler sees only train
    X_train_scaled = scaler.transform(X_train)
    X_test_scaled  = scaler.transform(X_test)
```

**Sklearn Pipelines** enforce the correct pattern automatically -- `fit` is only called on training data, and `transform` is applied consistently.

### Why Pipelines Matter in Production
- Single object to serialize and deploy
- Preprocessing and model always stay in sync
- No risk of applying wrong transformations at inference time

In [ ]:
# ============================================================
# Mixed-Type Dataset for ColumnTransformer Demo
# ============================================================
np.random.seed(42)
n = 1000

mixed_df = pd.DataFrame({
    # Numeric features
    'age':          np.random.normal(40, 12, n).clip(18, 80).astype(int),
    'income':       np.random.lognormal(10, 1, n),
    'credit_score': np.random.normal(680, 80, n).clip(300, 850).astype(int),
    'num_accounts': np.random.poisson(3, n),
    # Categorical features
    'employment':   np.random.choice(['Full-time', 'Part-time', 'Self-employed', 'Retired'], n),
    'education':    np.random.choice(['High School', 'Bachelor', 'Master', 'PhD'], n),
    'region':       np.random.choice(['North', 'South', 'East', 'West'], n),
})

# Inject some missing values
for col in ['income', 'credit_score', 'employment']:
    mask = np.random.rand(n) < 0.05
    mixed_df.loc[mask, col] = np.nan

# Target: loan default (binary)
logit = (
    -0.03 * mixed_df['age'].fillna(40)
    - 0.0000001 * mixed_df['income'].fillna(mixed_df['income'].median())
    + 0.001 * mixed_df['credit_score'].fillna(680)
    + 0.1 * mixed_df['num_accounts']
    + np.random.normal(0, 0.5, n)
)
mixed_df['default'] = (logit > logit.median()).astype(int)

print('Mixed dataset shape:', mixed_df.shape)
print(mixed_df.dtypes)
print(f'\nMissing values:\n{mixed_df.isnull().sum()}')
print(f'\nTarget distribution:\n{mixed_df["default"].value_counts()}')

In [ ]:
# ============================================================
# ColumnTransformer + Full Pipeline
# ============================================================
X_mixed = mixed_df.drop(columns=['default'])
y_mixed = mixed_df['default']

X_tr, X_te, y_tr, y_te = train_test_split(X_mixed, y_mixed, test_size=0.2,
                                           random_state=42, stratify=y_mixed)

# Identify numeric and categorical columns
num_cols = ['age', 'income', 'credit_score', 'num_accounts']
cat_cols = ['employment', 'education', 'region']

# Preprocessing sub-pipelines
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine into ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_cols),
    ('cat', cat_transformer, cat_cols)
])

# Full end-to-end pipeline
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier',   RandomForestClassifier(n_estimators=100, random_state=42))
])

print('Pipeline structure:')
print(full_pipeline)
print('\nStep names:', [step[0] for step in full_pipeline.steps])

In [ ]:
# ============================================================
# Train the Full Pipeline -- no manual preprocessing needed
# ============================================================
full_pipeline.fit(X_tr, y_tr)  # fit only on training data

# Predict on test set -- pipeline applies learned transformations
y_pred = full_pipeline.predict(X_te)
y_prob = full_pipeline.predict_proba(X_te)[:, 1]

acc  = accuracy_score(y_te, y_pred)
auc  = roc_auc_score(y_te, y_prob)
cv   = cross_val_score(full_pipeline, X_tr, y_tr, cv=5, scoring='roc_auc')

print(f'Test Accuracy : {acc:.4f}')
print(f'Test ROC AUC  : {auc:.4f}')
print(f'CV ROC AUC    : {cv.mean():.4f} +/- {cv.std():.4f}')
print('\nClassification Report:')
print(classification_report(y_te, y_pred))

In [ ]:
# ============================================================
# Pipeline Serialization with joblib
# ============================================================
model_dir = Path('./saved_models')
model_dir.mkdir(exist_ok=True)

pipeline_path = model_dir / 'loan_default_pipeline_v1.joblib'
joblib.dump(full_pipeline, pipeline_path)

file_size_kb = pipeline_path.stat().st_size / 1024
print(f'Pipeline saved to: {pipeline_path}')
print(f'File size: {file_size_kb:.1f} KB')

# Load and verify identical predictions
loaded_pipeline = joblib.load(pipeline_path)
y_pred_loaded   = loaded_pipeline.predict(X_te)

assert np.array_equal(y_pred, y_pred_loaded), 'Predictions should be identical!'
print('Loaded pipeline produces identical predictions.')
print('This single file contains ALL preprocessing + model -- safe to deploy.')

In [ ]:
# ============================================================
# Demonstrate Why Pipelines Prevent Data Leakage
# ============================================================
print('=== Data Leakage Demonstration ===')
print()
print('WRONG approach (leakage):')
print('-' * 40)
# Bad: scaler fitted on all data before split
scaler_bad = StandardScaler()
X_mixed_num = X_mixed[num_cols].fillna(X_mixed[num_cols].median())
X_all_scaled_bad = scaler_bad.fit_transform(X_mixed_num)       # LEAKS test stats
X_tr_bad = X_all_scaled_bad[:int(0.8*len(X_mixed))]
X_te_bad = X_all_scaled_bad[int(0.8*len(X_mixed)):]
print(f'  Scaler mean[income] = {scaler_bad.mean_[1]:.2f}  (computed on ALL data including test)')

print()
print('CORRECT approach (no leakage):')
print('-' * 40)
# Good: scaler fitted only on training data
scaler_good = StandardScaler()
X_tr_num = X_tr[num_cols].fillna(X_tr[num_cols].median())
X_te_num = X_te[num_cols].fillna(X_tr[num_cols].median())  # use train stats for imputation
scaler_good.fit(X_tr_num)                                    # fit only on train
X_tr_good = scaler_good.transform(X_tr_num)
X_te_good = scaler_good.transform(X_te_num)                  # apply train stats to test
print(f'  Scaler mean[income] = {scaler_good.mean_[1]:.2f}  (computed on train only)')

print()
print('Pipeline: handles this automatically.')
print('  pipeline.fit(X_train)     -> all steps fitted on train only')
print('  pipeline.predict(X_test)  -> transform using train-fitted stats')

In [ ]:
# ============================================================
# Dataset Versioning -- DVC Concept Implemented Manually
# DVC (Data Version Control) does this with git-like commands
# ============================================================

class SimpleDataVersioner:
    """
    Implements the core concept behind DVC:
    - Track dataset hashes instead of storing raw data in git
    - Record provenance: what data, when, how processed
    """

    def __init__(self, registry_path='./data_versions.json'):
        self.registry_path = registry_path
        self.registry = self._load()

    def _load(self):
        if Path(self.registry_path).exists():
            with open(self.registry_path) as f:
                return json.load(f)
        return {}

    def _save(self):
        with open(self.registry_path, 'w') as f:
            json.dump(self.registry, f, indent=2)

    def _hash_df(self, df):
        return hashlib.md5(
            pd.util.hash_pandas_object(df, index=True).values.tobytes()
        ).hexdigest()

    def register(self, df, version_tag, description='', source=''):
        content_hash = self._hash_df(df)
        entry = {
            'version':      version_tag,
            'hash':         content_hash,
            'rows':         len(df),
            'cols':         len(df.columns),
            'columns':      list(df.columns),
            'registered_at': datetime.now().isoformat(),
            'description':  description,
            'source':       source
        }
        self.registry[version_tag] = entry
        self._save()
        print(f'Registered dataset version: {version_tag}')
        print(f'  Hash : {content_hash}')
        print(f'  Shape: {len(df)} rows x {len(df.columns)} cols')
        return content_hash

    def verify(self, df, version_tag):
        if version_tag not in self.registry:
            print(f'Version {version_tag} not found in registry.')
            return False
        current_hash  = self._hash_df(df)
        expected_hash = self.registry[version_tag]['hash']
        if current_hash == expected_hash:
            print(f'Dataset {version_tag}: VERIFIED (hash matches)')
            return True
        else:
            print(f'Dataset {version_tag}: MISMATCH! Data may have changed.')
            print(f'  Expected: {expected_hash}')
            print(f'  Got:      {current_hash}')
            return False

    def history(self):
        print(f'\n{"Version":<20}{"Rows":<8}{"Hash":<10}{"Registered":<25}')
        print('-' * 63)
        for v, info in self.registry.items():
            print(f"{v:<20}{info['rows']:<8}{info['hash'][:8]:<10}{info['registered_at'][:19]:<25}")


versioner = SimpleDataVersioner()
versioner.register(X_tr, 'train_v1.0', description='Initial training set', source='loan_applications_2023Q1')
versioner.register(X_te, 'test_v1.0',  description='Hold-out test set',    source='loan_applications_2023Q1')

print()
versioner.verify(X_tr, 'train_v1.0')  # should pass

X_tr_modified = X_tr.copy()
X_tr_modified.iloc[0, 0] = -999        # corrupt one value
versioner.verify(X_tr_modified, 'train_v1.0')  # should fail

versioner.history()

---
## Section 3 -- Model Monitoring After Deployment

### Why Models Degrade in Production

A model trained last year on historical data faces a world that keeps changing:
- Customer behavior shifts (seasonal, economic)
- Data pipelines change upstream (new fields, encoding changes)
- The relationship between features and targets evolves

**Two distinct failure modes:**

| Failure Mode | Definition | Example |
|---|---|---|
| **Covariate Shift** | P(X) changes but P(Y|X) stays the same | Age distribution of applicants shifts younger |
| **Concept Drift** | P(Y|X) changes (the relationship changes) | Same credit profile, higher default rate during recession |

### Key Metrics to Monitor
1. **PSI** -- Population Stability Index (features)
2. **KL Divergence** -- statistical measure of distribution shift
3. **Prediction distribution** -- monitor model output drift
4. **Business metrics** -- precision, recall on labeled production data (when available)

In [ ]:
# ============================================================
# KL Divergence for Data Drift Detection
# ============================================================

def kl_divergence(p, q, epsilon=1e-10):
    """
    KL Divergence D_KL(P || Q) from bins/histograms.
    p and q are empirical probability distributions (must sum to 1).
    epsilon avoids log(0).
    """
    p = np.array(p, dtype=float) + epsilon
    q = np.array(q, dtype=float) + epsilon
    p /= p.sum()
    q /= q.sum()
    return float(np.sum(p * np.log(p / q)))


def js_divergence(p, q):
    """Jensen-Shannon Divergence -- symmetric version of KL, bounded [0, ln2]."""
    p = np.array(p, dtype=float)
    q = np.array(q, dtype=float)
    p /= p.sum()
    q /= q.sum()
    m = 0.5 * (p + q)
    return 0.5 * kl_divergence(p, m) + 0.5 * kl_divergence(q, m)


def compute_drift(train_series, prod_series, n_bins=20, feature_name='feature'):
    """Compute KL and JS divergence between training and production distributions."""
    combined = np.concatenate([train_series.dropna(), prod_series.dropna()])
    bins     = np.linspace(combined.min(), combined.max(), n_bins + 1)

    train_hist, _ = np.histogram(train_series.dropna(), bins=bins, density=True)
    prod_hist,  _ = np.histogram(prod_series.dropna(),  bins=bins, density=True)

    kl = kl_divergence(train_hist, prod_hist)
    js = js_divergence(train_hist, prod_hist)

    # Alert thresholds (common in industry)
    if kl < 0.1:
        status = 'OK'
    elif kl < 0.2:
        status = 'WARNING'
    else:
        status = 'ALERT'

    return {'feature': feature_name, 'kl_divergence': round(kl, 4),
            'js_divergence': round(js, 4), 'status': status}


# Simulate production data with varying degrees of drift
np.random.seed(99)
n_prod = 500

train_data = X_tr[num_cols].copy()

# Simulate three scenarios
no_drift_prod      = pd.DataFrame({c: train_data[c].dropna().sample(n_prod, replace=True).values for c in num_cols})
moderate_drift_prod = pd.DataFrame({
    'age':          np.random.normal(50, 15, n_prod).clip(18, 80),   # older customers
    'income':       np.random.lognormal(10.5, 0.9, n_prod),           # higher income
    'credit_score': np.random.normal(650, 90, n_prod).clip(300, 850), # slightly lower scores
    'num_accounts': np.random.poisson(4, n_prod)                      # more accounts
})
severe_drift_prod = pd.DataFrame({
    'age':          np.random.normal(25, 5, n_prod).clip(18, 80),    # much younger
    'income':       np.random.lognormal(9.0, 1.5, n_prod),            # lower, more variable
    'credit_score': np.random.normal(580, 100, n_prod).clip(300, 850),# lower scores
    'num_accounts': np.random.poisson(1, n_prod)                      # fewer accounts
})

print('Drift Analysis Results:')
print(f'{"Feature":<15}{"Scenario":<20}{"KL Div":<12}{"JS Div":<12}{"Status"}')
print('-' * 65)
for scenario_name, prod_df in [("No Drift", no_drift_prod),
                                ("Moderate Drift", moderate_drift_prod),
                                ("Severe Drift", severe_drift_prod)]:
    for col in num_cols:
        result = compute_drift(train_data[col], prod_df[col], feature_name=col)
        print(f"{col:<15}{scenario_name:<20}{result['kl_divergence']:<12}{result['js_divergence']:<12}{result['status']}")

In [ ]:
# ============================================================
# Population Stability Index (PSI)
# Used extensively in banking and credit risk modeling
# ============================================================

def compute_psi(expected, actual, n_bins=10, epsilon=1e-4):
    """
    PSI = sum( (Actual% - Expected%) * ln(Actual% / Expected%) )

    Interpretation:
      PSI < 0.10  : No significant shift
      0.10 - 0.25 : Moderate shift, investigate
      PSI > 0.25  : Major shift, model likely needs retraining
    """
    # Create bins from expected distribution
    breakpoints = np.percentile(expected, np.linspace(0, 100, n_bins + 1))
    breakpoints[0]  -= 1e-6
    breakpoints[-1] += 1e-6

    expected_pcts = []
    actual_pcts   = []

    for i in range(n_bins):
        lo, hi = breakpoints[i], breakpoints[i+1]
        exp_pct = np.sum((expected > lo) & (expected <= hi)) / len(expected)
        act_pct = np.sum((actual   > lo) & (actual   <= hi)) / len(actual)
        expected_pcts.append(max(exp_pct, epsilon))
        actual_pcts.append(max(act_pct, epsilon))

    psi_values = [(a - e) * np.log(a / e) for a, e in zip(actual_pcts, expected_pcts)]
    psi = sum(psi_values)

    if psi < 0.10:
        flag = 'STABLE'
    elif psi < 0.25:
        flag = 'MODERATE SHIFT'
    else:
        flag = 'MAJOR SHIFT'

    return psi, flag, expected_pcts, actual_pcts


print('Population Stability Index (PSI)')
print('Threshold: <0.10=Stable, 0.10-0.25=Moderate, >0.25=Major Shift')
print()
print(f'{"Feature":<15}{"Scenario":<20}{"PSI":<10}{"Flag"}')
print('-' * 60)
for scenario_name, prod_df in [("No Drift", no_drift_prod),
                                ("Moderate Drift", moderate_drift_prod),
                                ("Severe Drift", severe_drift_prod)]:
    for col in num_cols:
        psi, flag, _, _ = compute_psi(
            train_data[col].dropna().values,
            prod_df[col].dropna().values
        )
        print(f'{col:<15}{scenario_name:<20}{psi:<10.4f}{flag}')

In [ ]:
# ============================================================
# Visualize Drift for One Feature
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

scenarios = [
    ("No Drift",       no_drift_prod,       'green'),
    ("Moderate Drift", moderate_drift_prod, 'orange'),
    ("Severe Drift",   severe_drift_prod,   'red'),
]

bins = np.linspace(18, 80, 30)
for ax, (name, prod_df, color) in zip(axes, scenarios):
    ax.hist(train_data['age'].dropna(), bins=bins, density=True,
            alpha=0.5, color='steelblue', label='Training')
    ax.hist(prod_df['age'], bins=bins, density=True,
            alpha=0.5, color=color, label='Production')
    psi, flag, _, _ = compute_psi(train_data['age'].dropna().values,
                                   prod_df['age'].values)
    ax.set_title(f'{name}\nPSI={psi:.4f} ({flag})', fontsize=9)
    ax.set_xlabel('Age')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.suptitle('Data Drift Detection: Feature Distribution Comparison (Age)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Concept Drift Simulation
# Covariate shift vs concept drift -- key interview distinction
# ============================================================
np.random.seed(42)
time_steps = 12  # months

# Generate a dataset where:
# - Feature distribution stays the same (no covariate shift)
# - But the relationship P(Y|X) changes (concept drift)

monthly_data = []
np.random.seed(10)
for month in range(time_steps):
    n_month = 200
    x = np.random.normal(0, 1, n_month)  # same distribution every month

    # The RELATIONSHIP changes over time (concept drift)
    # Coefficient on x gradually shifts from positive to negative
    drift_coeff = 1.0 - (2.0 * month / time_steps)  # goes from +1 to -1
    logit = drift_coeff * x + np.random.normal(0, 0.5, n_month)
    y = (logit > 0).astype(int)

    monthly_data.append({
        'month': month,
        'x_mean': x.mean(),
        'x_std':  x.std(),
        'y_rate': y.mean(),
        'drift_coeff': drift_coeff
    })

drift_df = pd.DataFrame(monthly_data)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(drift_df['month'], drift_df['x_mean'], marker='o', color='steelblue')
axes[0].fill_between(drift_df['month'],
                      drift_df['x_mean'] - drift_df['x_std'],
                      drift_df['x_mean'] + drift_df['x_std'],
                      alpha=0.2, color='steelblue')
axes[0].set_title('Feature X: Mean +/- Std\n(No Covariate Shift)', fontsize=9)
axes[0].set_xlabel('Month')
axes[0].axhline(0, color='red', linestyle='--')

axes[1].plot(drift_df['month'], drift_df['drift_coeff'], marker='s', color='darkorange')
axes[1].set_title('True Coefficient of X\n(Concept Drift)', fontsize=9)
axes[1].set_xlabel('Month')
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_ylabel('Coefficient')

axes[2].plot(drift_df['month'], drift_df['y_rate'], marker='^', color='green')
axes[2].set_title('Observed Y Rate\n(Detectable Symptom of Concept Drift)', fontsize=9)
axes[2].set_xlabel('Month')
axes[2].set_ylabel('Positive Rate')

plt.suptitle('Concept Drift: P(Y|X) Changes Even Though P(X) Is Stable', fontweight='bold')
plt.tight_layout()
plt.show()

print('Covariate shift : P(X) changes. Same feature statistics in train vs prod? No.')
print('Concept drift   : P(Y|X) changes. Same features, different target relationship.')
print('Both can happen simultaneously -- most real production failures involve both.')

In [ ]:
# ============================================================
# Alerting System for Production Monitoring
# ============================================================

class ModelMonitor:
    """Production model monitoring with configurable alerting thresholds."""

    def __init__(self, model_name, thresholds=None):
        self.model_name = model_name
        self.thresholds = thresholds or {
            'psi_warning':   0.10,
            'psi_critical':  0.25,
            'kl_warning':    0.10,
            'kl_critical':   0.20,
            'accuracy_drop': 0.05,   # alert if accuracy drops by >5pp
            'pred_rate_pct': 0.20,   # alert if prediction rate changes >20%
        }
        self.baseline_metrics = None
        self.alert_log = []

    def set_baseline(self, X_baseline, y_baseline, pipeline):
        y_pred = pipeline.predict(X_baseline)
        self.baseline_metrics = {
            'accuracy':   accuracy_score(y_baseline, y_pred),
            'pred_rate':  y_pred.mean(),
            'feature_distributions': {
                col: X_baseline[col].dropna().values
                for col in X_baseline.select_dtypes(include=[np.number]).columns
            }
        }
        print(f'Baseline set: accuracy={self.baseline_metrics["accuracy"]:.4f}, '
              f'pred_rate={self.baseline_metrics["pred_rate"]:.4f}')

    def check_drift(self, X_prod):
        alerts = []
        for col, baseline_vals in self.baseline_metrics['feature_distributions'].items():
            if col not in X_prod.columns:
                continue
            psi, flag, _, _ = compute_psi(baseline_vals, X_prod[col].dropna().values)
            if psi > self.thresholds['psi_critical']:
                alerts.append({'level': 'CRITICAL', 'type': 'PSI', 'feature': col, 'value': round(psi, 4)})
            elif psi > self.thresholds['psi_warning']:
                alerts.append({'level': 'WARNING',  'type': 'PSI', 'feature': col, 'value': round(psi, 4)})
        return alerts

    def check_predictions(self, y_pred_prod):
        alerts = []
        if self.baseline_metrics is None:
            return alerts
        prod_rate = y_pred_prod.mean()
        base_rate = self.baseline_metrics['pred_rate']
        rel_change = abs(prod_rate - base_rate) / (base_rate + 1e-10)
        if rel_change > self.thresholds['pred_rate_pct']:
            alerts.append({'level': 'WARNING', 'type': 'PRED_RATE',
                           'detail': f'Rate changed {rel_change:.1%}: {base_rate:.3f} -> {prod_rate:.3f}'})
        return alerts

    def run_check(self, X_prod, pipeline):
        print(f'\n--- Monitor: {self.model_name} ---')
        all_alerts = []
        all_alerts.extend(self.check_drift(X_prod))
        y_pred_prod = pipeline.predict(X_prod)
        all_alerts.extend(self.check_predictions(y_pred_prod))
        if all_alerts:
            for a in all_alerts:
                print(f'  [{a["level"]}] {a.get("type","")} | {a.get("feature", a.get("detail", ""))} | value={a.get("value", "N/A")}')
        else:
            print('  All checks PASSED. No alerts.')
        self.alert_log.extend(all_alerts)
        return all_alerts


monitor = ModelMonitor('loan_default_v1')
monitor.set_baseline(X_te, y_te, full_pipeline)

X_te_num = X_te.select_dtypes(include=[np.number])
no_drift_full      = X_te.copy()
moderate_drift_full = X_te.copy()
moderate_drift_full['age']    = moderate_drift_prod['age'].values[:len(X_te)]
moderate_drift_full['income'] = moderate_drift_prod['income'].values[:len(X_te)]

print()
monitor.run_check(no_drift_full, full_pipeline)
monitor.run_check(moderate_drift_full, full_pipeline)

---
## Section 4 -- Model Deployment Patterns

Deployment is where ML meets software engineering. Senior candidates must understand the tradeoffs between different serving architectures.

### The Four Main Patterns

| Pattern | Latency | Throughput | When to Use |
|---|---|---|---|
| **REST API (online)** | Low (<100ms) | Medium | Real-time decisions |
| **Batch inference** | High (minutes/hours) | Very high | Offline scoring, reports |
| **Streaming** | Very low | High | Event-driven systems |
| **Edge deployment** | Near-zero | Low | Mobile, IoT |

### A/B Testing for Models -- The Statistical Framework

Never deploy a new model without a rigorous A/B test. The key question: *"Is the performance difference real or noise?"*

In [ ]:
# ============================================================
# REST API Pattern -- Flask pseudocode + explanation
# (Does not actually run a server; illustrates the pattern)
# ============================================================

flask_api_code = '''
# Production REST API for ML Model Serving (Flask)
# Pattern: preload model at startup, serve predictions per request

from flask import Flask, request, jsonify
import joblib
import pandas as pd
import logging
import time

app    = Flask(__name__)
logger = logging.getLogger(__name__)

# Load model ONCE at startup -- not per request
MODEL_PATH   = '/models/loan_default_pipeline_v2.joblib'
MODEL_VERSION = 'v2.1.0'
pipeline = joblib.load(MODEL_PATH)

@app.route('/health', methods=['GET'])
def health():
    """Kubernetes liveness / readiness probe endpoint."""
    return jsonify({'status': 'ok', 'model_version': MODEL_VERSION})

@app.route('/predict', methods=['POST'])
def predict():
    start_time = time.time()
    try:
        data = request.get_json(force=True)
        df   = pd.DataFrame([data['features']])

        # Validate required fields
        required = ['age', 'income', 'credit_score', 'num_accounts',
                    'employment', 'education', 'region']
        missing = [f for f in required if f not in df.columns]
        if missing:
            return jsonify({'error': f'Missing fields: {missing}'}), 400

        # Get prediction and probability
        prediction = int(pipeline.predict(df)[0])
        probability = float(pipeline.predict_proba(df)[0][1])
        latency_ms = (time.time() - start_time) * 1000

        # Log for monitoring
        logger.info({'event': 'prediction', 'input': data['features'],
                     'prediction': prediction, 'probability': probability,
                     'latency_ms': latency_ms, 'model_version': MODEL_VERSION})

        return jsonify({
            'prediction':   prediction,       # 0 = no default, 1 = default
            'probability':  probability,       # probability of class 1
            'model_version': MODEL_VERSION,
            'latency_ms':   round(latency_ms, 2)
        })

    except Exception as e:
        logger.error(f'Prediction error: {e}')
        return jsonify({'error': str(e)}), 500

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=8080, workers=4)
'''

print('Flask REST API Pattern:')
print(flask_api_code)
print('Key design decisions:')
print('  1. Model loaded once at startup (not per request -- huge latency saving)')
print('  2. /health endpoint for orchestrator probes')
print('  3. Input validation before prediction')
print('  4. Every prediction logged for drift monitoring')
print('  5. Model version included in response (traceable)')

In [ ]:
# ============================================================
# Batch vs Real-Time Inference Tradeoff Analysis
# ============================================================

tradeoff_table = {
    'Dimension':         ['Latency',       'Throughput',    'Cost',          'Freshness',      'Complexity', 'Use Case'],
    'Real-Time (REST)':  ['<100ms',         'Moderate',     'Higher (always on)', 'Instant',   'High',      'Fraud detection, recs'],
    'Batch (scheduled)': ['Minutes-Hours',  'Very High',    'Lower (spot VMs)', 'Stale OK',    'Low',       'Monthly scoring, reports'],
    'Streaming (Kafka)': ['<1s',            'High',         'Medium',         'Near-real-time', 'Very High', 'Event-driven pipelines'],
}

df_tradeoff = pd.DataFrame(tradeoff_table)
print('Inference Pattern Tradeoffs:')
print(df_tradeoff.to_string(index=False))

print()
print('Batch Inference Advantages:')
batch_pros = [
    '- Precompute scores for all users nightly -> instant lookup at request time',
    '- Use spot/preemptible instances (80% cost saving)',
    '- Easier debugging: process is deterministic and logged',
    '- Simple infrastructure: cron + Spark/pandas, no server to maintain'
]
for p in batch_pros:
    print(p)

print()
print('Real-Time Inference Advantages:')
rt_pros = [
    '- Always uses latest user state (last 5 minutes of behavior)',
    '- Handles new users with no history',
    '- Required for latency-sensitive use cases (ad bidding <50ms)',
    '- Can incorporate request context (device, geo, time)'
]
for p in rt_pros:
    print(p)

In [ ]:
# ============================================================
# A/B Testing for Models -- Full Statistical Framework
# ============================================================

def simulate_ab_test(model_a_accuracy, model_b_accuracy,
                     n_samples=1000, n_simulations=1000,
                     significance_level=0.05):
    """
    Simulate an A/B test between two models.
    Uses a two-proportion z-test.
    Returns: test results including p-value and decision.
    """
    np.random.seed(42)

    # Generate outcomes for each model
    outcomes_a = np.random.binomial(1, model_a_accuracy, n_samples)
    outcomes_b = np.random.binomial(1, model_b_accuracy, n_samples)

    # Two-proportion z-test
    p_a = outcomes_a.mean()
    p_b = outcomes_b.mean()
    n_a = len(outcomes_a)
    n_b = len(outcomes_b)

    # Pooled proportion under null hypothesis (no difference)
    p_pool = (outcomes_a.sum() + outcomes_b.sum()) / (n_a + n_b)
    se = np.sqrt(p_pool * (1 - p_pool) * (1/n_a + 1/n_b))
    z_stat = (p_b - p_a) / se
    p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))  # two-tailed

    # Effect size (Cohen's h for proportions)
    h = 2 * np.arcsin(np.sqrt(p_b)) - 2 * np.arcsin(np.sqrt(p_a))

    # Minimum detectable effect at 80% power
    mde = 1.96 * se + 0.84 * se  # z_alpha/2 + z_beta

    # Confidence interval for difference
    se_diff = np.sqrt(p_a*(1-p_a)/n_a + p_b*(1-p_b)/n_b)
    ci_low  = (p_b - p_a) - 1.96 * se_diff
    ci_high = (p_b - p_a) + 1.96 * se_diff

    decision = 'DEPLOY MODEL B' if p_value < significance_level and p_b > p_a else 'KEEP MODEL A'

    return {
        'n_samples_per_arm': n_samples,
        'model_a_accuracy':  round(p_a, 4),
        'model_b_accuracy':  round(p_b, 4),
        'absolute_lift':     round(p_b - p_a, 4),
        'relative_lift_pct': round((p_b - p_a) / p_a * 100, 2),
        'z_statistic':       round(z_stat, 4),
        'p_value':           round(p_value, 6),
        'significant':       p_value < significance_level,
        'cohens_h':          round(h, 4),
        'ci_95':             (round(ci_low, 4), round(ci_high, 4)),
        'decision':          decision
    }


# Scenario 1: Practically significant difference
print('Scenario 1: Model B is clearly better (91% vs 87%)')
result1 = simulate_ab_test(0.87, 0.91, n_samples=1000)
for k, v in result1.items():
    print(f'  {k:<25}: {v}')

print()
print('Scenario 2: Marginal difference (88% vs 87% -- same model, noise)')
result2 = simulate_ab_test(0.87, 0.88, n_samples=1000)
for k, v in result2.items():
    print(f'  {k:<25}: {v}')

print()
print('Scenario 3: Larger sample -- detect smaller effects')
result3 = simulate_ab_test(0.87, 0.88, n_samples=10000)
for k, v in result3.items():
    print(f'  {k:<25}: {v}')

In [ ]:
# ============================================================
# Visualize A/B Test Power Analysis
# ============================================================

sample_sizes = [100, 250, 500, 1000, 2000, 5000, 10000]
true_effects = [0.01, 0.02, 0.05]
base_acc     = 0.87

fig, ax = plt.subplots(figsize=(10, 5))

colors = ['steelblue', 'darkorange', 'green']
for eff, color in zip(true_effects, colors):
    powers = []
    for n in sample_sizes:
        p_a = base_acc
        p_b = base_acc + eff
        p_pool = 0.5 * (p_a + p_b)
        se = np.sqrt(p_pool * (1 - p_pool) * 2 / n)
        ncp = abs(p_b - p_a) / se  # non-centrality parameter
        power = 1 - stats.norm.cdf(1.96 - ncp)  # one-sided approx
        powers.append(power)
    ax.plot(sample_sizes, powers, marker='o', color=color,
            label=f'True effect = {eff:.0%} absolute')

ax.axhline(0.80, color='black', linestyle='--', linewidth=1.5, label='80% power threshold')
ax.set_xscale('log')
ax.set_xlabel('Sample Size per Arm')
ax.set_ylabel('Statistical Power')
ax.set_title('A/B Test Power Analysis: Sample Size vs Detectable Effect')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print('Key insight: to detect a 1% lift you need ~10x more samples than for a 5% lift.')
print('This is why large-scale A/B testing requires significant traffic volume.')

In [ ]:
# ============================================================
# Shadow Mode and Canary Deployment
# ============================================================

class ShadowDeployment:
    """
    Shadow mode: new model runs in parallel but its predictions
    are NOT used to serve the user. Compare outputs offline.
    Zero user impact, full production traffic validation.
    """

    def __init__(self, production_model, shadow_model):
        self.production_model = production_model
        self.shadow_model     = shadow_model
        self.shadow_log       = []

    def predict(self, X):
        """Serve production prediction; shadow prediction logged silently."""
        prod_pred   = self.production_model.predict(X)
        shadow_pred = self.shadow_model.predict(X)

        # Log agreement for offline analysis
        agreement = np.mean(prod_pred == shadow_pred)
        self.shadow_log.append({
            'n_requests':    len(X),
            'prod_pred_rate': prod_pred.mean(),
            'shadow_pred_rate': shadow_pred.mean(),
            'agreement_rate': agreement
        })
        return prod_pred  # only production served to user

    def shadow_report(self):
        df = pd.DataFrame(self.shadow_log)
        print('Shadow Deployment Report:')
        print(df.describe().round(4))
        overall_agree = df['agreement_rate'].mean()
        print(f'\nOverall agreement: {overall_agree:.2%}')
        if overall_agree > 0.95:
            print('Models agree strongly. Safe to promote shadow -> production.')
        else:
            print('Significant disagreement. Investigate before promotion.')


class CanaryDeployment:
    """
    Canary: route small fraction of real traffic to new model.
    Roll back instantly if metrics degrade.
    """

    def __init__(self, production_model, canary_model, canary_fraction=0.05):
        self.production_model = production_model
        self.canary_model     = canary_model
        self.canary_fraction  = canary_fraction
        self.metrics_log      = {'production': [], 'canary': []}

    def predict(self, X, true_labels=None):
        """
        Route traffic: canary_fraction goes to new model, rest to production.
        """
        n = len(X)
        canary_mask = np.random.rand(n) < self.canary_fraction

        predictions = np.empty(n, dtype=int)
        predictions[~canary_mask] = self.production_model.predict(X.iloc[~canary_mask])
        if canary_mask.any():
            predictions[canary_mask] = self.canary_model.predict(X.iloc[canary_mask])

        if true_labels is not None:
            prod_acc   = accuracy_score(true_labels[~canary_mask], predictions[~canary_mask])
            canary_acc = accuracy_score(true_labels[canary_mask],  predictions[canary_mask]) if canary_mask.any() else None
            self.metrics_log['production'].append(prod_acc)
            if canary_acc is not None:
                self.metrics_log['canary'].append(canary_acc)

        return predictions

    def should_rollback(self, threshold=0.02):
        if not self.metrics_log['canary']:
            return False
        prod_mean   = np.mean(self.metrics_log['production'])
        canary_mean = np.mean(self.metrics_log['canary'])
        degradation = prod_mean - canary_mean
        print(f'Production mean accuracy: {prod_mean:.4f}')
        print(f'Canary mean accuracy    : {canary_mean:.4f}')
        print(f'Degradation             : {degradation:.4f}')
        if degradation > threshold:
            print('ROLLBACK TRIGGERED: canary performance below threshold.')
            return True
        else:
            print('Canary healthy. Consider increasing traffic fraction.')
            return False


# Demo with our trained models
best_result = max(results, key=lambda r: r['accuracy'])
worse_result = min(results, key=lambda r: r['accuracy'])

print('=== Shadow Deployment Demo ===')
shadow = ShadowDeployment(best_result['best_model'], worse_result['best_model'])
for _ in range(5):
    shadow.predict(X_test_scaled)
shadow.shadow_report()

print()
print('=== Canary Deployment Demo ===')
canary = CanaryDeployment(best_result['best_model'], worse_result['best_model'], canary_fraction=0.10)
for _ in range(20):
    idx = np.random.choice(len(X_test_scaled), 50)
    batch = pd.DataFrame(X_test_scaled[idx], columns=wine.feature_names)
    canary.predict(batch, true_labels=y_test[idx])

canary.should_rollback(threshold=0.02)

---
## Section 5 -- Production ML Best Practices

### The Hidden Technical Debt in ML Systems

The famous Google paper ("Hidden Technical Debt in Machine Learning Systems") found that actual ML code is a small fraction of a real production ML system:

```
+----------------------------------------------------------+
|  Data Collection  |  Feature Engineering  |  Monitoring |
|  Data Verification|  Config Management    |  Serving     |
|  Process Mgmt     |  Resource Mgmt        |  Analysis    |
|            [ ML Code (the small box) ]                   |
+----------------------------------------------------------+
```

The hardest problems in production ML are NOT modeling -- they are data management, reproducibility, and operational reliability.

In [ ]:
# ============================================================
# Feature Store -- Simple Implementation
# A feature store centralizes feature computation and serving
# ============================================================

class SimpleFeatureStore:
    """
    Feature Store -- centralized registry for features.
    Solves the training-serving skew problem:
    compute features once, use consistently in training and inference.
    """

    def __init__(self, store_name='main_feature_store'):
        self.store_name   = store_name
        self.feature_defs = {}   # feature_name -> transformation function
        self.materialized = {}   # entity_id -> {feature_name -> value}
        self.feature_meta = {}   # metadata about each feature

    def register_feature(self, name, transform_fn, description='',
                         owner='', sla_hours=24):
        """
        Register a feature with its transformation function.
        The same function used in training and serving.
        """
        self.feature_defs[name] = transform_fn
        self.feature_meta[name] = {
            'description':  description,
            'owner':        owner,
            'sla_hours':    sla_hours,
            'created_at':   datetime.now().isoformat(),
            'version':      '1.0'
        }
        print(f'Registered feature: {name}')

    def compute_features(self, entity_df, entity_id_col='entity_id'):
        """
        Compute all registered features for a DataFrame of entities.
        Returns a DataFrame of entity_id + all features.
        """
        results = {entity_id_col: entity_df[entity_id_col].values}
        for fname, fn in self.feature_defs.items():
            results[fname] = fn(entity_df)
        result_df = pd.DataFrame(results)

        # Materialize (cache) for fast online lookup
        for _, row in result_df.iterrows():
            eid = row[entity_id_col]
            self.materialized[eid] = {
                k: v for k, v in row.items() if k != entity_id_col
            }

        return result_df

    def get_online_features(self, entity_id):
        """
        Low-latency online serving: return precomputed features for one entity.
        In production this would hit Redis/DynamoDB, not a Python dict.
        """
        if entity_id not in self.materialized:
            raise KeyError(f'Entity {entity_id} not found in feature store')
        return self.materialized[entity_id]

    def catalog(self):
        print(f'\nFeature Store: {self.store_name}')
        print(f'{"Feature":<30}{"Owner":<15}{"SLA (hrs)":<12}{"Description"}')
        print('-' * 75)
        for name, meta in self.feature_meta.items():
            print(f"{name:<30}{meta['owner']:<15}{meta['sla_hours']:<12}{meta['description'][:30]}")


# Build a feature store for the loan dataset
fs = SimpleFeatureStore('loan_feature_store')

# Register features with their computation logic
fs.register_feature(
    'income_log',
    lambda df: np.log1p(df['income'].fillna(df['income'].median())),
    description='Log-transformed income',
    owner='data-science',
    sla_hours=24
)
fs.register_feature(
    'age_bucket',
    lambda df: pd.cut(df['age'], bins=[0, 25, 35, 50, 65, 100],
                      labels=['18-25', '26-35', '36-50', '51-65', '65+']).astype(str),
    description='Age group bucket',
    owner='data-science',
    sla_hours=24
)
fs.register_feature(
    'credit_income_ratio',
    lambda df: df['credit_score'].fillna(680) / (df['income'].fillna(1) + 1),
    description='Credit score per income unit',
    owner='risk-team',
    sla_hours=1
)
fs.register_feature(
    'has_excellent_credit',
    lambda df: (df['credit_score'].fillna(0) >= 750).astype(int),
    description='Binary: credit_score >= 750',
    owner='risk-team',
    sla_hours=1
)

# Compute features for training data
X_tr_with_id = X_tr.reset_index().rename(columns={'index': 'entity_id'})
feature_matrix = fs.compute_features(X_tr_with_id, entity_id_col='entity_id')

print(f'\nFeature matrix shape: {feature_matrix.shape}')
print(feature_matrix.head(3).to_string())

# Online lookup
eid = X_tr_with_id.iloc[0]['entity_id']
online_features = fs.get_online_features(eid)
print(f'\nOnline features for entity {eid}:')
for k, v in online_features.items():
    print(f'  {k}: {v}')

fs.catalog()

In [ ]:
# ============================================================
# Model Versioning and Rollback Strategy
# ============================================================

class ProductionModelManager:
    """
    Manages the lifecycle of models in production:
    - Versioned deployments
    - Instant rollback
    - Deployment history
    """

    def __init__(self):
        self.deployment_history = []
        self.current_version    = None
        self.current_model      = None

    def deploy(self, model, version, metrics, deployed_by='auto'):
        record = {
            'version':     version,
            'model':       model,
            'metrics':     metrics,
            'deployed_at': datetime.now().isoformat(),
            'deployed_by': deployed_by,
            'status':      'active'
        }
        # Mark previous as replaced
        for r in self.deployment_history:
            if r['status'] == 'active':
                r['status'] = 'replaced'

        self.deployment_history.append(record)
        self.current_version = version
        self.current_model   = model
        print(f'Deployed model version: {version} by {deployed_by}')

    def rollback(self, n=1):
        """Roll back to the n-th previous version."""
        active_versions = [r for r in self.deployment_history if r['version'] != self.current_version]
        if len(active_versions) < n:
            print('Not enough history to roll back.')
            return False

        target = active_versions[-(n)]
        print(f'ROLLBACK: reverting from {self.current_version} to {target["version"]}')
        self.deploy(target['model'], target['version'] + '-rollback',
                    target['metrics'], deployed_by='rollback-system')
        return True

    def deployment_log(self):
        print(f'\n{"Version":<25}{"Status":<12}{"Accuracy":<12}{"Deployed At"}')
        print('-' * 65)
        for r in self.deployment_history:
            acc = r['metrics'].get('accuracy', 'N/A')
            print(f"{r['version']:<25}{r['status']:<12}{str(acc):<12}{r['deployed_at'][:19]}")


mgr = ProductionModelManager()

# Simulate deployment history
for r in results[:3]:
    mgr.deploy(
        model=r['best_model'],
        version=f"v{results.index(r)+1}.0.0",
        metrics={'accuracy': round(r['accuracy'], 4)},
        deployed_by='ml-pipeline'
    )

print()
mgr.deployment_log()

print()
print('Simulating production incident -- rolling back...')
mgr.rollback(n=1)
mgr.deployment_log()

In [ ]:
# ============================================================
# Production ML Monitoring Checklist
# ============================================================

checklist = {
    'Data Quality': [
        ('Missing value rate per feature',        True),
        ('Out-of-range values (schema validation)', True),
        ('Feature cardinality (new unseen categories)', True),
        ('Data pipeline SLA (freshness)',          True),
    ],
    'Data Drift': [
        ('PSI > 0.10 per feature (weekly)',        True),
        ('KL divergence on prediction distribution', True),
        ('Input feature summary statistics (mean/std)', True),
        ('Covariate shift detection (KS test)',    False),
    ],
    'Model Performance': [
        ('Accuracy/AUC on labeled production data', True),
        ('Prediction rate over time',              True),
        ('Latency p50/p95/p99',                   True),
        ('Error rate on API endpoint',             True),
    ],
    'Business Metrics': [
        ('Downstream KPI correlation',             True),
        ('Cost per false positive/negative',       False),
        ('A/B test results tracked',               True),
    ],
    'Operational': [
        ('Model artifact version pinned',          True),
        ('Rollback procedure tested',              True),
        ('Alerting configured',                    True),
        ('On-call rotation for model incidents',   False),
    ]
}

print('Production ML Monitoring Checklist')
print('=' * 55)
for category, items in checklist.items():
    done = sum(1 for _, v in items if v)
    print(f'\n{category} ({done}/{len(items)} implemented):')
    for item, status in items:
        mark = '[x]' if status else '[ ]'
        print(f'  {mark} {item}')

total_done  = sum(v for cat in checklist.values() for _, v in cat)
total_items = sum(len(cat) for cat in checklist.values())
print(f'\nOverall: {total_done}/{total_items} items implemented ({total_done/total_items:.0%})')

---
## Section 5 -- 15 MLOps Interview Questions with Detailed Answers

These are the most commonly asked MLOps questions in senior ML interviews at companies like Google, Meta, Airbnb, Stripe, and Databricks.

### Q1: How do you detect data drift in production?

**Answer:**
Data drift is when the statistical distribution of input features changes between training and production.

**Detection methods (in order of sophistication):**

1. **Summary statistics monitoring** -- track mean, std, min/max per feature. Simple but catches obvious shifts.

2. **PSI (Population Stability Index)** -- industry standard in banking. Computes expected vs actual bucket distributions. PSI < 0.10 = stable, 0.10-0.25 = monitor, > 0.25 = retrain.

3. **KL Divergence / JS Divergence** -- information-theoretic measures. KL is asymmetric; JS is symmetric and bounded. Good for continuous features.

4. **Two-sample statistical tests** -- Kolmogorov-Smirnov test for continuous, Chi-squared for categorical. Gives a p-value for the null hypothesis of same distribution.

5. **Model-based drift detection** -- train a classifier to distinguish training from production data. If it achieves high AUC, distributions differ.

**Practical monitoring architecture:**
- Log all predictions and feature values in production
- Compute drift metrics daily/hourly vs a rolling baseline window
- Alert when thresholds exceeded; auto-trigger retraining pipeline

### Q2: What is a feature store and why do you need one?

**Answer:**
A feature store is a centralized system for computing, storing, and serving ML features consistently across training and inference.

**The problem it solves -- Training-Serving Skew:**
- Data Science computes feature X in Python/pandas for training
- Engineering reimplements feature X in Java/SQL for production
- Subtle differences cause model degradation (wrong imputation, different bucketing)
- Feature store: one definition, used in both contexts

**Components:**
1. **Feature Registry** -- catalog of feature definitions, owners, schemas
2. **Offline Store** (e.g., S3 + Parquet) -- for batch training, historical lookups
3. **Online Store** (e.g., Redis, DynamoDB) -- for low-latency serving (<10ms)
4. **Feature Pipeline** -- computes features and writes to both stores

**When you need one:**
- Multiple models share the same features (avoid redundant computation)
- Training-serving skew causing unexplained production degradation
- Feature reuse across teams at scale

**Examples:** Feast (open source), Tecton, AWS SageMaker Feature Store, Google Vertex Feature Store

### Q3: How would you A/B test two models?

**Answer:**
A proper model A/B test requires:

**1. Define the metric and hypothesis**
- Primary metric: business KPI (conversion rate, precision on fraud, etc.)
- H0: Model B performance = Model A performance
- H1: Model B performance > Model A performance (one-sided for deployment decision)

**2. Determine sample size before starting**
- Use power analysis: decide minimum detectable effect, desired power (80%), significance level (5%)
- Common mistake: stopping as soon as you see p < 0.05 (peeking problem)

**3. Traffic split and randomization**
- Split by entity (user/customer ID), not by request -- same entity always sees same model
- Avoid Simpson's paradox: stratify by important segments

**4. Run for sufficient duration**
- At least 1-2 full weekly cycles to capture day-of-week effects
- Do NOT stop early

**5. Analysis**
- Two-proportion z-test for binary metrics
- t-test or Mann-Whitney for continuous metrics
- Bonferroni correction if testing multiple metrics

**6. Decision criteria**
- Statistical significance (p < 0.05) AND practical significance (effect size matters)
- Check for heterogeneous effects in subgroups

**Guardrails:** always monitor latency and error rates; stop if degradation detected

### Q4: What is concept drift vs covariate shift?

**Answer:**

**Covariate Shift:** P(X) changes but P(Y|X) stays the same.
- The feature distribution changes but the underlying relationship is stable
- Example: a model trained on 30-50 year olds deployed to a younger audience
- Solution: importance weighting (reweight training data to match production distribution)

**Concept Drift:** P(Y|X) changes -- the relationship between features and target changes.
- Same feature values now predict different outcomes
- Example: fraud patterns change as fraudsters adapt to detection
- Example: income -> default relationship changes during economic recession
- Solution: periodic retraining, sliding window training data

**Prior Shift:** P(Y) changes -- the class distribution changes.
- Example: fraud rate spikes during holiday season
- Solution: recalibrate decision thresholds, adjust class weights

**How to distinguish:**
- If model accuracy drops AND feature distributions changed -> likely covariate shift
- If model accuracy drops but feature distributions look stable -> likely concept drift
- In practice, both often happen simultaneously

**Detection:**
- Covariate shift: PSI/KL on features
- Concept drift: monitor model accuracy on labeled production data (requires labels, which are delayed)

### Q5: Walk me through how you would deploy a model to production.

**Answer (structured):**

**1. Pre-deployment validation**
- Offline evaluation on held-out test set and recent production-like data
- Fairness analysis across demographic subgroups
- Adversarial testing / edge cases
- Load testing: can the serving infrastructure handle peak QPS?

**2. Staging environment**
- Deploy to staging with production-like traffic replay
- Verify feature computation matches training
- Check latency SLAs (p99 < 100ms is common)

**3. Shadow deployment**
- Run new model in parallel with production for 1-2 weeks
- Compare predictions: high disagreement warrants investigation
- Zero risk to users

**4. Canary deployment**
- Route 5% of traffic to new model
- Monitor business metrics, latency, error rates in real-time
- Auto-rollback if metrics degrade beyond threshold
- Gradually increase: 5% -> 20% -> 50% -> 100%

**5. Full deployment + monitoring**
- Set up drift monitoring and alerting
- Document model card (training data, performance, limitations)
- Configure retraining triggers

**Key artifacts to version:** model binary, feature schema, preprocessing config, evaluation results, training data version

### Q6: How do you handle class imbalance in production models?

**Answer:**

**Data-level techniques:**
- Oversampling minority class (SMOTE -- synthetic minority oversampling)
- Undersampling majority class (random or informed, e.g., Tomek links)
- Class weights in loss function (sklearn: `class_weight='balanced'`)

**Evaluation -- what NOT to do:**
- Never use accuracy as primary metric with imbalanced data (a model predicting all negatives gets 99% accuracy on 1% fraud dataset)
- Use: Precision-Recall AUC, F1, ROC AUC, Average Precision

**Threshold tuning:**
- Default 0.5 threshold is rarely optimal
- Choose threshold based on business cost matrix: cost(false positive) vs cost(false negative)
- Plot Precision-Recall curve; choose operating point based on business requirement

**In production:**
- Monitor the actual class distribution over time (prior shift)
- Recalibrate thresholds periodically as base rate changes
- Track false positive and false negative rates separately in monitoring

### Q7: What is model calibration and why does it matter in production?

**Answer:**
A model is **well-calibrated** if its predicted probabilities match empirical frequencies:
- If a model outputs 0.7 for 1000 samples, ~700 should actually be positive

**Why it matters:**
- Decision thresholds and expected value calculations depend on calibrated probabilities
- Risk management, pricing, and ranking all require reliable probabilities
- An uncalibrated model that outputs 0.9 might actually correspond to 60% frequency

**Common calibration issues:**
- Random Forests tend to be overconfident (probabilities pushed to extremes)
- Naive Bayes tends to be underconfident (probabilities pushed to center)
- Neural networks can be overconfident without temperature scaling

**Calibration methods:**
- **Platt Scaling:** fit a logistic regression on model outputs (requires holdout set)
- **Isotonic Regression:** non-parametric, more flexible than Platt
- **Temperature Scaling:** for neural networks, divide logits by learned temperature T

**In sklearn:**
```python
from sklearn.calibration import CalibratedClassifierCV
calibrated = CalibratedClassifierCV(base_model, method='platt', cv=5)
calibrated.fit(X_train, y_train)
```

**Monitoring:** track calibration in production using reliability diagrams on labeled production data

### Q8: How do you prevent data leakage in ML pipelines?

**Answer:**
Data leakage means information from the test set or future data influences model training, leading to overly optimistic offline metrics and poor production performance.

**Types of leakage:**

1. **Preprocessing leakage** -- fitting transformers (scaler, imputer, encoder) on all data before splitting. Solution: use sklearn Pipelines; always fit only on training data.

2. **Target leakage** -- including features that are consequences of the target, not causes. Example: including `claim_paid` as a feature when predicting `insurance_fraud` -- every fraud case has a paid claim.

3. **Temporal leakage** -- using future information to predict past events. Always split time series data chronologically, never randomly.

4. **Group leakage** -- train/test split puts related examples on different sides. Example: same customer in train and test; model memorizes customer behavior.

5. **Label leakage** -- labels computed using information not available at prediction time.

**Best practices:**
- Use sklearn Pipelines (prevents preprocessing leakage automatically)
- For time series: use TimeSeriesSplit, never KFold
- For grouped data: use GroupKFold
- Feature review: for each feature, ask "would this be available at prediction time?"
- Cross-validate your data splitting strategy, not just model performance

### Q9: What are the key differences between batch and real-time ML systems?

**Answer:**

**Batch Inference (offline scoring)**
- Precompute predictions for all entities and store results
- User lookup at request time is just a database read (fast!)
- Can use expensive models without latency constraints
- Stale: predictions are N hours old
- Tools: Spark, Airflow DAGs, scheduled Lambda functions
- Use cases: product recommendations refreshed nightly, risk scores for next-day decisions

**Real-Time Inference (online serving)**
- Compute prediction at request time using live features
- Requires low-latency infrastructure (model server, feature cache)
- Freshness: uses current user state
- Expensive to scale; requires auto-scaling, load balancing
- Tools: Flask/FastAPI, TorchServe, TensorFlow Serving, Triton
- Use cases: fraud detection (must be instant), ad bidding (<50ms SLA)

**Hybrid approach (common in practice):**
- Precompute expensive features (batch) -> store in feature store
- At request time: fetch precomputed features + compute fast real-time features
- Run lightweight model in real-time on combined features
- Example: Netflix recommendations use batch-computed user embeddings + real-time context

**The right question:** what is the decision latency requirement and how stale can predictions be?

### Q10: How do you version ML models and why does it matter?

**Answer:**
Model versioning tracks exactly what artifact is serving which traffic, and enables rollback, audit, and reproducibility.

**What to version:**
1. Model binary (weights, architecture, hyperparameters)
2. Feature schema (exact features, types, expected ranges)
3. Preprocessing pipeline (same joblib file that was trained)
4. Training data version (hash or DVC pointer)
5. Training code (git commit SHA)
6. Evaluation results (metrics, confusion matrix, on which data)

**Version numbering convention:**
- `MAJOR.MINOR.PATCH` -- e.g., `v2.1.3`
- Major: incompatible feature schema change (requires redeployment)
- Minor: new training run with same features (retrained on new data)
- Patch: threshold change, calibration update

**MLflow approach:**
- Every training run logged with all params/metrics
- Promote runs to Model Registry stages: None -> Staging -> Production
- Load model by stage: `mlflow.pyfunc.load_model('models:/name/Production')`

**Critical rule:** the model artifact and the preprocessing pipeline MUST be versioned together. Deploying a new model with old preprocessing (or vice versa) is a common production failure mode.

**Rollback SLA:** good teams target <5 minutes to rollback to previous version in an incident.

### Q11: Explain the ML lifecycle and where automation helps.

**Answer:**
The ML lifecycle (MLOps full loop):

```
1. Data Ingestion -> 2. Feature Engineering -> 3. Training
       ^                                              |
       |                                              v
8. Monitoring <- 7. Production Serving <- 4. Validation
                                          |
                                          5. Registry
                                          |
                                          6. Deployment
```

**Where automation (CI/CD for ML) helps:**

- **Triggered retraining:** when drift detected OR on schedule, auto-trigger training pipeline
- **Automated evaluation gates:** new model only promoted if it beats baseline on holdout set
- **Continuous integration for data:** validate schema, distributions, and row count before training
- **Shadow deployment automation:** new models auto-shadow for N days before promotion
- **Rollback automation:** if prediction error rate spikes, auto-rollback to last known good version

**Tools:**
- Kubeflow Pipelines / Vertex AI Pipelines for orchestration
- MLflow for tracking and registry
- Great Expectations for data validation
- Grafana + Prometheus for monitoring
- Seldon / BentoML / Triton for serving

**Maturity model:**
- Level 0: manual, ad-hoc training
- Level 1: automated training pipelines but manual deployment
- Level 2: full CI/CD -- auto-train, auto-validate, auto-deploy

### Q12: How do you handle model failures in production?

**Answer:**

**Types of failures:**
1. **Hard failures** -- exception thrown, prediction not returned
2. **Soft failures** -- prediction returned but incorrect (silent degradation)

**Hard failure mitigation:**
- Fallback to simpler model (logistic regression always available as backup)
- Return cached last prediction for the entity
- Return default safe value (conservative score)
- Fail open vs fail closed depending on business logic

**Incident response process:**
1. Auto-alert fires (latency SLA breach, error rate spike, or drift alert)
2. On-call engineer assesses severity
3. Rollback decision: if root cause unknown, rollback first, investigate second
4. Root cause analysis (RCA): was it the model? data pipeline? feature store?
5. Fix and re-deploy with additional tests
6. Post-mortem: what monitoring would have caught this earlier?

**Circuit breaker pattern:**
```python
if error_rate > threshold or latency_p99 > sla:
    # Route all traffic to fallback model
    use_fallback = True
```

**Key lesson:** silent degradation is harder than hard failures. Without labeled production data, you may not know your model is wrong for days. Monitoring prediction distributions (not just latency/errors) is essential.

### Q13: What is shadow mode deployment and when do you use it?

**Answer:**
Shadow mode (also called dark launch or mirror traffic) means the new model processes real production traffic and generates predictions, but those predictions are NOT served to users or used in decisions.

**Purpose:**
- Validate the new model behaves correctly on real production traffic before any user impact
- Catch bugs in feature pipelines, preprocessing, encoding
- Compare prediction distributions between old and new model
- Measure production latency under real load

**When to use it:**
- Major model changes (new architecture, new features)
- After long periods without production traffic
- When the cost of a wrong prediction is high (medical, financial)

**What to measure in shadow mode:**
1. Prediction agreement rate (old vs new) -- high disagreement warrants investigation
2. Production latency of new model (p50, p95, p99)
3. Error rate (exceptions, null outputs)
4. Prediction distribution shift
5. Feature processing correctness

**Limitations:**
- Cannot measure business impact (predictions are not acted upon)
- Requires routing infrastructure to duplicate requests
- Expensive: doubles compute cost during shadow period

**Typical duration:** 1-2 weeks, or until sufficient volume and confidence achieved

### Q14: How do you ensure reproducibility in ML experiments?

**Answer:**
Reproducibility means given the same code, data, and configuration, you get the same result.

**Five layers of reproducibility:**

1. **Code reproducibility**
   - Git commit SHA for every experiment
   - Pinned dependency versions (`requirements.txt` or `conda.yml` with exact versions)
   - No global state, no hardcoded paths

2. **Data reproducibility**
   - Version datasets (DVC, Delta Lake versioning, or content hashing)
   - Store exact train/test split indices (or the random seed + split strategy)
   - Never overwrite raw data -- append only

3. **Environment reproducibility**
   - Docker containers with pinned base images
   - CUDA version pinned for GPU experiments

4. **Experiment configuration**
   - All hyperparameters, feature lists, preprocessing steps logged (MLflow)
   - No magic numbers in code -- everything in a config file

5. **Randomness control**
   - Set all random seeds: `numpy.random.seed`, `random.seed`, `torch.manual_seed`
   - Log the seeds used
   - Note: some operations are non-deterministic even with seeds (multi-GPU training)

**Practical checklist:**
- Can someone else checkout your git repo, download your data version, run your pipeline, and get within epsilon of your reported metrics? If yes: reproducible.

### Q15: How do you evaluate the business impact of an ML model?

**Answer:**
Model metrics (AUC, F1) are proxies. The real question is: does the model create business value?

**Framework: connect model metrics to business outcomes**

1. **Define the decision problem clearly**
   - What action does the model enable or automate?
   - What is the cost of each error type? (false positive, false negative)

2. **Expected Value calculation**
   ```
   EV = TP * value(correct action) - FP * cost(wrong action) - FN * cost(missed opportunity)
   ```
   Example (fraud detection):
   ```
   EV per prediction = TP * $500 (fraud blocked) - FP * $10 (customer friction) - FN * $500 (fraud missed)
   ```

3. **Comparison baseline**
   - Compare to current state: no model, simple rules, previous model
   - Incremental lift, not absolute performance

4. **A/B test for causal attribution**
   - Offline metrics can be misleading
   - Only a proper A/B test with business KPIs confirms actual impact

5. **Total cost of ownership**
   - Model development + maintenance cost
   - Infrastructure cost (GPU serving, feature store)
   - Risk cost (model failures, bias/fairness issues)
   - Data labeling cost

**Red flag question to ask:** "What happens if the model outputs random predictions?" If business metrics barely change, the model isn't in a critical decision path -- reconsider the deployment.

In [ ]:
# ============================================================
# Section 5 Summary Visualization
# ============================================================
import textwrap

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: PSI thresholds reference
psi_ranges  = ['< 0.10\nStable', '0.10 - 0.25\nModerate', '> 0.25\nMajor Shift']
psi_heights = [0.10, 0.15, 0.25]
psi_colors  = ['green', 'orange', 'red']
bars = axes[0, 0].bar(psi_ranges, psi_heights, color=psi_colors, edgecolor='white', linewidth=0.5)
axes[0, 0].set_ylabel('PSI Value')
axes[0, 0].set_title('PSI Thresholds for Data Drift Monitoring')
for bar, h in zip(bars, psi_heights):
    axes[0, 0].text(bar.get_x() + bar.get_width()/2, h + 0.005, str(h), ha='center', fontsize=9)

# Plot 2: Deployment pattern risk vs complexity
patterns   = ['Direct\nDeploy', 'Rolling\nDeploy', 'Blue/Green', 'Canary\n5%', 'Shadow\nMode']
risk       = [5, 3, 2, 1, 0]
complexity = [1, 2, 3, 4, 5]
axes[0, 1].scatter(complexity, risk, s=200, c=['red','orange','yellow','lightgreen','green'],
                   zorder=3, edgecolors='grey')
for i, p in enumerate(patterns):
    axes[0, 1].annotate(p, (complexity[i], risk[i]), textcoords='offset points',
                         xytext=(8, 0), fontsize=8)
axes[0, 1].set_xlabel('Deployment Complexity')
axes[0, 1].set_ylabel('Production Risk')
axes[0, 1].set_title('Deployment Patterns: Risk vs Complexity')
axes[0, 1].grid(alpha=0.3)

# Plot 3: MLOps Maturity Levels
levels  = ['Level 0\nManual', 'Level 1\nAuto Training', 'Level 2\nFull CI/CD']
scores  = [1, 2.5, 5]
colors3 = ['tomato', 'gold', 'mediumseagreen']
bars3   = axes[1, 0].barh(levels, scores, color=colors3, edgecolor='white')
axes[1, 0].set_xlabel('Automation Level')
axes[1, 0].set_title('MLOps Maturity Model')
labels3 = ['No automation,\njupyter notebooks',
           'Pipeline automation,\nmanual deployment',
           'Auto train, validate,\ndeploy, monitor']
for bar, label in zip(bars3, labels3):
    axes[1, 0].text(0.1, bar.get_y() + bar.get_height()/2, label, va='center', fontsize=7)

# Plot 4: Sample size requirements for A/B testing
baseline_acc = 0.87
effects      = np.array([0.005, 0.01, 0.02, 0.03, 0.05, 0.10])
# Approximation: n = (z_alpha/2 + z_beta)^2 * 2*p*(1-p) / delta^2
z_sum = 1.96 + 0.84  # alpha=0.05, power=0.80
samples_needed = (z_sum ** 2) * 2 * baseline_acc * (1 - baseline_acc) / (effects ** 2)

axes[1, 1].loglog(effects * 100, samples_needed, marker='o', color='steelblue', linewidth=2)
axes[1, 1].set_xlabel('Minimum Detectable Effect (%)')
axes[1, 1].set_ylabel('Required Sample Size per Arm')
axes[1, 1].set_title('A/B Test Sample Size vs Effect Size\n(power=80%, alpha=5%)')
axes[1, 1].grid(alpha=0.3, which='both')
axes[1, 1].axhline(10000, color='red', linestyle='--', linewidth=1, label='10k samples')
axes[1, 1].legend(fontsize=8)

plt.suptitle('MLOps Reference Charts', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Final Summary: Key Concepts Recap
# ============================================================

summary = {
    'Section 1 - Experiment Tracking': [
        'Use MLflow (or equivalent) to log params, metrics, artifacts for every run',
        'Model Registry manages lifecycle: None -> Staging -> Production -> Archived',
        'Never rely on notebook memory to track what worked; log everything'
    ],
    'Section 2 - Pipelines & Reproducibility': [
        'sklearn Pipeline = preprocessor + model in one serializable object',
        'ColumnTransformer handles mixed types (numeric impute+scale, cat impute+encode)',
        'Pipelines prevent data leakage by enforcing fit-only-on-train',
        'Version datasets with content hashing; version code with git SHA'
    ],
    'Section 3 - Model Monitoring': [
        'PSI: < 0.10 stable, 0.10-0.25 investigate, > 0.25 retrain',
        'Covariate shift: P(X) changes. Concept drift: P(Y|X) changes',
        'Monitor prediction distribution, not just feature distributions',
        'Labels in production are delayed -- monitor proxies in the meantime'
    ],
    'Section 4 - Deployment Patterns': [
        'Shadow mode: zero risk, validate before any production impact',
        'Canary: small % of real traffic, auto-rollback if metrics degrade',
        'A/B test: define metric + sample size BEFORE starting; avoid peeking',
        'Batch vs real-time: tradeoff between freshness and infrastructure cost'
    ],
    'Section 5 - Production Best Practices': [
        'Feature store: one definition for training and serving eliminates skew',
        'Model version must include preprocessing pipeline (always deploy together)',
        'Rollback SLA target: < 5 minutes to previous known-good version',
        'Connect model metrics to business metrics via expected value calculation'
    ]
}

print('=' * 65)
print('NOTEBOOK SUMMARY: MLOps and Production ML')
print('=' * 65)
for section, points in summary.items():
    print(f'\n{section}:')
    for p in points:
        print(f'  * {p}')

print()
print('=' * 65)
print('You are now prepared for senior ML engineering interviews')
print('on the topics of MLOps and production machine learning.')
print('=' * 65)